# 03 — Stability of the loading subspace

Is the FFM loading subspace stable over 2014–2025? Loadings are only identified up to rotation,
so stability is studied geometrically, through the **Grassmann distance** between loading subspaces
estimated on rolling windows, and then tested formally.

Requires the shared objects saved by `01_data_eda_ffm_rolling.ipynb` (`data/shared/`).

## Contents

1. [Imports & shared objects](#Imports-&-shared-objects)
2. [Geometry — Grassmann distance](#Geometry)
   - [Per pair (vs 2014)](#Grassmann-distance-per-pair-(vs-2014))
   - [Window to window](#Grassmann-distance-window-to-window)
   - [2014 reference window](#Grassmann-distance-ref-2014)
   - [CUSUM](#CUSUM) and [Bai-Perron](#Bai-Perron)
3. [Simulated null distribution of $d_G$](#Simulation-$d_G$)
4. [Goodness of fit — rolling R²](#Quality-Fit)
5. [Loadings evolution](#Loadings-Evolution)
6. [Dette & Kutta break tests](#Dette-&-Kutta)

# Import

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
from functools import partial

In [ ]:
# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf

In [ ]:
# ── Data processing & factor model ───────────────────────────────────────────
from data_processing import resample_5min, log_returns_5min, trading_day_matrix, panel_to_dict
from ffm_transform import log_transform
from ffm_smoothing import smoothing_for_ffm, extract_smoothing_params
from ffm_fx_real import run_ffm, plot_raw_one_day
from factor_model_fcts import (
    build_panel_from_matrices, compute_daily_avg_vol, avg_vol_from_matrices,
    check_residual_common_structure, ffm_per_period,
)
from functions import plot_heatmap

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
from ffm_eda import run_eda, missing_by_year_pair, get_stable_pairs
from ffm_eda_focused import run_focused_eda

In [ ]:
# ── Forecasting ───────────────────────────────────────────────────────────────
from ffm_forecasting import (
    test_adf, test_kpss, gph_test, test_arch,
    fractional_diff, test_stationarity,
    plot_acf_pacf,
    test_ljung_box, test_ljung_box_func,
    forecast_ar, forecast_arima, forecast_garch,
    har_regression, plot_forecast_har,
    select_garch_order, select_garch_variant,
    plot_all_forecasts,
    predict_operator_h, plot_forecast_idiosyncratic,
    reconstruct_forecast,
    plot_forecast_vs_observed,
    decompose_forecast_error,
    print_error_decomposition,
    plot_error_decomposition,
    compare_models_error,
    print_models_comparison,
    plot_models_comparison,
    make_benchmark_forecast,
    select_p_far, forecast_idio_har,
    forecast_factor_rw, 
    forecast_idio_rw, run_model_comparison, 
    print_model_comparison, 
    plot_model_comparison
)

In [ ]:
# ── Stability ─────────────────────────────────────────────────────────────────
from ffm_stability import (
    grassman_rolling_window_ref, grassman_rolling_window,
    plot_grassman, plot_loadings_evolution,
    plot_loadings_mean_evolution, plot_r2_rolling,
    grassman_rolling_by_pair,
    analyze_negative_spikes,
    regression_grassmann_vol, plot_distance_distribution,
    loading_heatmap,
    compute_cusum, plot_cusum,
    bai_perron_distances, plot_bai_perron_distances,
    compute_cusum_ma5,
    hamed_rao_test, plot_hamed_rao,
    hamed_rao_rolling, plot_hamed_rao_rolling,
    bai_perron_distances_hac,
)

In [ ]:
from dg_simulation import run_simulation_dg, plot_simulation_dg, run_bootstrap_dg, plot_bootstrap_dg

In [ ]:
%load_ext autoreload
%autoreload 2

# Imports & shared objects

Load the objects computed and saved by `01_data_eda_ffm_rolling.ipynb`.

In [ ]:
# ── Load shared objects (saved by 01_data_eda_ffm_rolling) ────────────────────
import joblib

shared_dir = "../data/shared/"

ffm              = joblib.load(f"{shared_dir}ffm.pkl")
vol_ffm_all      = joblib.load(f"{shared_dir}vol_ffm_all.pkl")
vol_matrices_all = joblib.load(f"{shared_dir}vol_matrices_all.pkl")
avg_vol          = joblib.load(f"{shared_dir}avg_vol.pkl")
FX_all           = joblib.load(f"{shared_dir}FX_all.pkl")

print(f"ffm loaded  — T={ffm['U_hat'].shape[1]} days, N={len(ffm['pairs'])} pairs")
print(f"vol_ffm_all — {len(vol_ffm_all)} pairs")

# Geometry

### Grassmann distance per pair (vs 2014)

In [ ]:
grass_by_pair = grassman_rolling_by_pair(
    data        = vol_ffm_all,
    window_size = 6,
    step_months        = 1,
    func = lambda x, mu_ref=None: run_ffm(x, k_max=5, r_forced=1, silent=True, mu_ref=mu_ref),
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
    ref_start   = '2014-01-01',
)

## Grassmann distance window to window

In [ ]:
from functools import partial
run_ffm_r1 = partial(run_ffm, r_forced=1, silent= True)

dg_comparison = grassman_rolling_window( # compare each window with the previous one
        vol_ffm_all, 
        window_size = 6 ,  
        step = 6 , 
        func = run_ffm_r1, 
        start_date = '2014-01-01', 
        end_date = '2025-07-01'
        )

In [ ]:
plot_grassman(dg_comparison, avg_vol, 6, 30, reference=True) 

In [ ]:
plot_distance_distribution(dg_comparison, label='6M - 30d_ ww')

In [ ]:
result_cmp, cp_date, cv = compute_cusum(dg_comparison, confidence=0.95)
plot_cusum(result_cmp, cp_date, cv, window_label='6M - 30d_ ww')
#dg_comparison, label='6M - 30d_ ww'

In [ ]:
bp_cmp= bai_perron_distances(
    dg_comparison,
    col        = 'distance_grass',
    date_col   = 'start',
    max_breaks = 10,
    h_frac     = 0.15,
    criterion  = 'bic',
)
plot_bai_perron_distances(bp_cmp)

## Grassmann distance ref 2014

### r=1 

In [ ]:
period = [6]
step = [30]

for p in period:
    for s in step:
        print(f"Computing window={p}M, step={s}d")
        if p == 3: end = '2025-10-01'
        if p == 6: end = '2025-07-01'
        if p == 12: end = '2025-01-01'
        results = grassman_rolling_window_ref(
            vol_ffm_all, 
            window_size = p, 
            step = s, 
            start_date = '2014-01-01', 
            end_date = end 
        )
        results.to_parquet(f"../data/results/grass_dist_{p}_{s}_mu.parquet", engine='pyarrow')



In [ ]:
from ffm_stability import grassmann_distance_matrix, plot_grassmann_heatmap

In [ ]:

%load_ext autoreload
%autoreload 2

In [ ]:
dist_matrix, labels = grassmann_distance_matrix(
    vol_ffm_all,
    window_size=6,
    step=6,
    start_date='2014-01-01',
    end_date='2025-12-31'
)

plot_grassmann_heatmap(
    dist_matrix, labels,
    save_path='../figures/STABILITY/grassmann_heatmap_pairwise.png'
)

In [ ]:
print(np.max(dist_matrix))
print(dg_ref_6M_30d.nlargest(1, 'distance_grass'))

In [ ]:
print(dist_matrix[0])

In [ ]:
print(dg_ref_6M_30d.head())

In [ ]:
print(dg_ref_6M_30d.shape)

#### XM Yd

##### 3M 7d 

In [ ]:
dg_ref_3M_7d = pd.read_parquet("../data/results/grass_dist_3_7.parquet", engine='pyarrow')

In [ ]:
plot_grassman(dg_ref_3M_7d, avg_vol, 3, 7, reference=True) 

In [ ]:
plot_distance_distribution(dg_ref_3M_7d, label='3M - 30d')

##### 3M 30d

In [ ]:
dg_ref_3M_30d = pd.read_parquet("../data/results/grassmann_ref_3M_30d.parquet", engine='pyarrow')

In [ ]:
plot_grassman(dg_ref_3M_30d, avg_vol, 3, 30, reference=True) 

In [ ]:
plot_distance_distribution(dg_ref_3M_30d, label='3M - 30d')

##### 6M 7d

In [ ]:
dg_ref_6M_7d=  pd.read_parquet("../data/results/grass_dist_6_7.parquet", engine='pyarrow')

In [ ]:
plot_grassman(dg_ref_6M_7d, avg_vol, 6, 7, reference=True) 

In [ ]:
plot_distance_distribution(dg_ref_6M_7d, label='6M - 7d')

##### 6M 30d

In [ ]:
dg_ref_6M_30d = pd.read_parquet("../data/results/grass_dist_6_30_mu.parquet", engine='pyarrow')

In [ ]:
print(dg_ref_6M_30d.nlargest(1, 'distance_grass'))

In [ ]:
print(dg_ref_6M_30d.nlargest(5, 'distance_grass'))

In [ ]:
plot_grassman(dg_ref_6M_30d, avg_vol, 6, 30, reference=True) 

In [ ]:
# single plot
plot_grassman(dg_ref_6M_30d, avg_vol, 6, 30, reference=True, show_dynamics=False)


In [ ]:
plot_distance_distribution(dg_ref_6M_30d, label='6M - 30d')

In [ ]:
test_adf(dg_ref_6M_30d['distance_grass'])

In [ ]:
plot_acf_pacf(dg_ref_6M_30d['distance_grass'])

##### 12M 7d

In [ ]:
dg_ref_12M_7d = pd.read_parquet("../data/results/grassmann_ref_12M_7d.parquet", engine='pyarrow')

In [ ]:
plot_grassman(dg_ref_12M_7d, avg_vol, 12, 7, reference=True) 

In [ ]:
plot_distance_distribution(dg_ref_12M_7d, label='12M - 7d')

##### 12M 30d

In [ ]:
dg_ref_12M_30d = pd.read_parquet("../data/results/grassmann_ref_12M_30d.parquet", engine='pyarrow')

In [ ]:
plot_grassman(dg_ref_12M_30d, avg_vol, 12, 30, reference=True) 

In [ ]:
plot_distance_distribution(dg_ref_12M_30d, label='12M - 30d')

#### CUSUM

In [ ]:
result_6M_30d, cp_date, cv = compute_cusum(dg_ref_6M_30d, confidence=0.95)
plot_cusum(result_6M_30d, cp_date, cv, window_label='6M / 30d')

In [ ]:
result_6M_30d, cp_date, cv = compute_cusum_ma5(dg_ref_6M_30d, confidence=0.95)
plot_cusum(result_6M_30d, cp_date, cv, window_label='6M / 30d')

In [ ]:
# ── Global test (whole series) ────────────────────────────────────────────────
for label, dg in [
    #('3M / 7d',  dg_ref_3M_7d),
    #('6M / 7d',  dg_ref_6M_7d),
    ('6M / 30d', dg_ref_6M_30d),
]:
    print(f'\n--- {label} ---')
    res = hamed_rao_test(dg, col='distance_grass', date_col='start', alpha=0.05)
    plot_hamed_rao(res)


In [ ]:
# ── Rolling Hamed-Rao (3-year window, 6-month step) ──────────────────────────
roll = hamed_rao_rolling(
    dg_ref_6M_30d,
    col    = 'distance_grass',
    date_col = 'start',
    window = 12,
    step   = 6,
    alpha  = 0.05,
)
print(roll[['window_start','window_end','n','z','p','trend']].to_string(index=False))

plot_hamed_rao_rolling(
    roll,
    dist_df  = dg_ref_6M_30d,
    col      = 'distance_grass',
    date_col = 'start',
    alpha    = 0.05,
    window   = 36,
    step     = 6,
)

#### Bai-Perron 

In [ ]:
bp_6M_30d = bai_perron_distances(
    dg_ref_6M_30d,
    col        = 'distance_grass',
    date_col   = 'start',
    max_breaks = 10,
    h_frac     = 0.15,
    criterion  = 'bic',
)
plot_bai_perron_distances(bp_6M_30d)

In [ ]:
bp_6M_30d = bai_perron_distances_hac(
    dg_ref_6M_30d,
    col        = 'distance_grass',
    date_col   = 'start',
    max_breaks = 10,
    h_frac     = 0.05,
    criterion  = 'bic',
)
plot_bai_perron_distances(bp_6M_30d)

In [ ]:
print(dg_ref_6M_30d['distance_grass'].mean())

### r=4

In [ ]:
from functools import partial
run_ffm_r2 = partial(run_ffm, r_forced=2, silent= True)

In [ ]:
period = [6]
step = [30]

for p in period:
    for s in step:
        print(f"Computing window={p}M, step={s}d")
        if p == 3: end = '2025-10-01'
        if p == 6: end = '2025-07-01'
        if p == 12: end = '2025-01-01'
        results = grassman_rolling_window_ref(
            vol_ffm_all, 
            window_size = p, 
            step = s, 
            func = run_ffm_r2, 
            start_date = '2014-01-01', 
            end_date = end 
        )
        results.to_parquet(f"../data/results/grass_dist_{p}_{s}_mu_2r.parquet", engine='pyarrow')



In [ ]:
dg_r2_6_30 = pd.read_parquet("../data/results/grass_dist_6_30_mu_2r.parquet", engine='pyarrow')

In [ ]:
plot_grassman(dg_r2_6_30, avg_vol, 6, 30, reference=True) 

In [ ]:
plot_distance_distribution(dg_r4_6_30, label='6M - 30d')

In [ ]:
result_6M_30d_r4, cp_date, cv = compute_cusum(dg_r4_6_30, confidence=0.95)
plot_cusum(result_6M_30d_r4, cp_date, cv, window_label='6M / 30d')

In [ ]:
bp_6M_30d_4r = bai_perron_distances_hac(
    dg_r4_6_30,
    col        = 'distance_grass',
    date_col   = 'start',
    max_breaks = 11,
    h_frac     = 0.05,
    criterion  = 'lwz',
)
plot_bai_perron_distances(bp_6M_30d_4r)

## Simulation $d_G$

### r=1 

#### Covariance simulation

In [ ]:
dg_simulation = run_simulation_dg(vol_ffm_all, start_date='2014-01-02', end_date='2014-07-01', S=132, B=1000, r=1)

In [ ]:
pd.DataFrame({'distance_grass': dg_simulation}).to_parquet(f"../data/results/dg_simulation.parquet", engine='pyarrow')

In [ ]:
dg_simulation = pd.read_parquet("../data/results/dg_simulation.parquet", engine='pyarrow')
plot_simulation_dg(dg_simulation)

#### Bootstrap $d_G$

In [ ]:
dg_bootstrap = run_bootstrap_dg(
    data         = vol_ffm_all,
    start_date   = '2014-01-02',
    end_date     = '2014-07-01',
    S            = 132,
    B            = 1000,
    r            = 1,
    block_length = 5,
)

In [ ]:
pd.DataFrame({'distance_grass': dg_bootstrap}).to_parquet(
    "../data/results/dg_bootstrap_null.parquet", engine='pyarrow'
)

In [ ]:
#observed_mean = dg_ref_6M_30d['distance_grass'].mean()
plot_bootstrap_dg(dg_bootstrap)

### r =4

#### Covariance simulation

In [ ]:
dg_simulation_r4 = run_simulation_dg(vol_ffm_all, start_date='2014-01-02', end_date='2014-07-01', S=132, B=1000, r=4)

In [ ]:
pd.DataFrame({'distance_grass': dg_simulation_r4}).to_parquet(f"../data/results/dg_simulation_r4.parquet", engine='pyarrow')

In [ ]:
dg_simulation_r4 = pd.read_parquet("../data/results/dg_simulation_r4.parquet", engine='pyarrow')
plot_simulation_dg(dg_simulation_r4)

#### Bootstrap simulation

In [ ]:
dg_bootstrap_r4 = run_bootstrap_dg(
    data         = vol_ffm_all,
    start_date   = '2014-01-02',
    end_date     = '2014-07-01',
    S            = 132,
    B            = 1000,
    r            = 4,
    block_length = 5,
)

In [ ]:
pd.DataFrame({'distance_grass': dg_bootstrap_r4}).to_parquet(
    "../data/results/dg_bootstrap_r4.parquet", engine='pyarrow'
)

In [ ]:
dg_bootstrap_r4 = pd.read_parquet("../data/results/dg_bootstrap_r4.parquet", engine='pyarrow')
plot_bootstrap_dg(dg_bootstrap_r4)

# Quality Fit

### Rolling R² per pair

#### $\hat{r} = 1$ forced

In [ ]:
r2_results = plot_r2_rolling(
    data        = vol_ffm_all,
    window_size = 6,
    step        = 30,
    func        = lambda x: run_ffm(x, k_max=5, r_forced=1, silent=True),
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
    r='1'
)

#### $\hat{r}=2$ forced

In [ ]:
r2_results = plot_r2_rolling(
    data        = vol_ffm_all,
    window_size = 6,
    step        = 30,
    func        = lambda x: run_ffm(x, k_max=5, r_forced=2, silent=True),
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
    r='2'
)

#### Not forced

In [ ]:
r2_results = plot_r2_rolling(
    data        = vol_ffm_all,
    window_size = 6,
    step        = 30,
    func        = lambda x: run_ffm(x, k_max=5, silent=True),
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
    r='not_fixed'
)

# Loadings Evolution

In [ ]:
# compare the loadings across 5 windows
windows = [
    ('2014-10-01', '2015-04-01'),   # CHF floor removal
    ('2016-06-01', '2016-12-01'),   # Brexit 
    ('2018-07-01', '2019-01-01'),   # TRY crisis
    ('2020-01-01', '2020-07-01'),   # COVID
    ('2022-02-01', '2022-08-01'),   # Ukraine / Fed hikes
    
]

highlight_pairs = [
    # Majors USD (6)
    'eurusd',    # EUR + USD
    'gbpusd',    # GBP + USD
    'usdjpy',    # JPY + USD
    'eurchf',    # CHF + EUR
    'audusd',    # AUD + USD
    'usdcad',    # CAD + USD

    # Crosses without USD (4)
    'audnzd',    # AUD + NZD  ← only pair with NZD
    #'cadchf',    # CAD + CHF  ← redundant
    'sgdjpy',    # SGD + JPY  ← only pair with SGD
    'eurgbp',    # EUR + GBP  ← GBP cross without USD

    # EUR EM Europe (3)
    'eurnok',    # NOK
    'eursek',    # SEK
    'eurczk',    # CZK

    # USD EM (5)
    'usdpln',    # PLN
    'usdhuf',    # HUF
    'usdtry',    # TRY
    'usdmxn',    # MXN
    'zarjpy',    # ZAR
    'usdcnh'     # CNH
]


plot_loadings_evolution(
    vol_matrices    = vol_ffm_all,
    run_ffm         = lambda x, mu_ref=None: run_ffm(x, k_max=5, silent=True,  mu_ref=mu_ref),
    windows         = windows,
    r_plot          = 2,
    highlight_pairs = highlight_pairs,
    add_title = 'temp'
)

In [ ]:
remaining_pairs = [p for p in FX_all if p.lower() not in 
                   [h.lower() for h in highlight_pairs]]

plot_loadings_evolution(
    vol_matrices    = vol_ffm_all,
    run_ffm         = lambda x, mu_ref=None: run_ffm(x, k_max=5, silent=True, r_forced=1, mu_ref=mu_ref),
    windows         = windows,
    r_plot          = 1,
    highlight_pairs = remaining_pairs,
    add_title='2014_other'
)

### Mean Loadings Evolution (rolling)

In [ ]:
run_ffm_r1 = partial(run_ffm, r_forced=1, silent= True)

mean_loadings_arr, dates = plot_loadings_mean_evolution(
    data        = vol_ffm_all,
    window_size = 6,
    step        = 30,
    func        = run_ffm_r1,
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
    r_plot      = 2,
)

### Loadings heatmap

In [ ]:
run_ffm_r1 = partial(run_ffm, r_forced=1, silent=True)

from ffm_stability import loading_heatmap

df_heat = loading_heatmap(
    data        = vol_ffm_all,
    func        = run_ffm_r1,
    ref_start   = '2014-01-01',
    end_date    = '2025-07-01',
    window_size = 6,
    step_months = 1,
    mode = 'ref' #'prev'
)

# Dette & Kutta

Structural-break tests on the eigensystem of the stacked FFM panel.

Reference: Dette & Kutta (2021), *Electronic Journal of Statistics*.

The panel is stacked: Y ∈ ℝ^{T × NM} where N = number of pairs, M = number of intraday intervals.
All functions assume centred data (cf. eq. 2.1 of the paper).

In [ ]:
from ffm_stability import plot_eigensystem_diff_rolling

for p in [6]:
    plot_eigensystem_diff_rolling(
        vol_ffm_all,
        window_size=p,
        step_months=1,
        start_date='2014-01-01',
        end_date='2025-01-01',
    )


## 1. Panel construction

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd

from ffm_fx_real  import build_panel_from_matrices
from dette_kutta  import (
    stack_panel,
    estimate_change_point,
    estimate_change_point_eigenvalue,
    estimate_change_point_eigenfunction,
    eigenvalue_test,
    eigenfunction_test,
    calibrate_delta_v,
    calibrate_delta_tau,
)

X_3d, days, tau, pairs = build_panel_from_matrices(vol_ffm_all, min_coverage=0.8)
Y, delta = stack_panel(X_3d)

print(f"Panel: T={Y.shape[0]} days  D={Y.shape[1]} (={len(pairs)} pairs × {X_3d.shape[2]} intervals)")


## 2. Global estimator — full covariance operator

CUSUM on the Frobenius norm of the full operator (Eq. 2.6).
Detects the dominant break across all directions.

In [ ]:
cp_global = estimate_change_point(Y, eps=0.05, dates=days)
print(f"Break global : {str(cp_global['break_date'])[:10]}  (θ̂ = {cp_global['theta_hat']:.3f},  k₀ = {cp_global['k0']})")


## 3. Specific estimators — eigenvalue and eigenfunction

They maximise, respectively:
- `f_ev(k) = [k(N-k)/N²] · (τ₁^(1)(k) − τ₁^(2)(k))²`
- `f_ef(k) = [k(N-k)/N²] · ‖v₁^(1)(k) − v₁^(2)(k)‖²_L²`

Grid `step=20` (~1 month) to limit computation time.

In [ ]:
cp_ev = estimate_change_point_eigenvalue(Y, j=1, eps=0.05, step=20, dates=days)
cp_ef = estimate_change_point_eigenfunction(Y, j=1, eps=0.05, step=20, dates=days)

print(f"Break eigenvalue   : {str(cp_ev['break_date'])[:10]}  (θ̂ = {cp_ev['theta_hat']:.3f},  k₀ = {cp_ev['k0']})")
print(f"Break eigenfunction: {str(cp_ef['break_date'])[:10]}  (θ̂ = {cp_ef['theta_hat']:.3f},  k₀ = {cp_ef['k0']})")


## 4. Eigenvalue test — break 2016-07-15 (post-Brexit)

τ₁ drops from 0.367 to 0.215 around Brexit: −42% of explained variance.
We test H₀ : (τ₁^(1) − τ₁^(2))² ≤ 0 with the break fixed at the estimated date.

In [ ]:
res_ev_brexit = eigenvalue_test(
    Y,
    j          = 1,
    Delta_tau  = 0.0,
    alpha      = 0.05,
    K          = 20,
    break_date = cp_ev["break_date"],
    plot       = True,
    pair_name  = "panel_ev_brexit",
    dates      = days,
    delta=delta
)

print(f"Break : {str(cp_ev['break_date'])[:10]}")
print(f"τ^(1)_1 = {res_ev_brexit['tau1_j']:.5f}   τ^(2)_1 = {res_ev_brexit['tau2_j']:.5f}   Δ = {res_ev_brexit['tau1_j'] - res_ev_brexit['tau2_j']:.5f}")
print(f"Ê  = {res_ev_brexit['E_hat']:.6f}   V̂ = {res_ev_brexit['V_hat']:.6f}")
print(f"stat = {res_ev_brexit['stat']:.3f}   q = {res_ev_brexit['q_crit']:.3f}   p-value = {res_ev_brexit['p_value']:.4f}")
print(f"Reject H₀ : {res_ev_brexit['reject']}")


## 5. Eigenfunction test — break 2018-11-16 (US–China trade war)

We test H₀ : ‖v₁^(1) − v₁^(2)‖²_L² ≤ 0 with the break fixed at the estimated date.

In [ ]:
res_ef_trade = eigenfunction_test(
    Y,
    j          = 1,
    Delta_v    = 0.0,
    alpha      = 0.05,
    K          = 20,
    break_date = cp_ef["break_date"],
    plot       = True,
    pair_name  = "panel_ef_trade",
    dates      = days,
    delta=delta
)

print(f"Break : {str(cp_ef['break_date'])[:10]}")
print(f"D̂  = {res_ef_trade['D_hat']:.5f}   Û = {res_ef_trade['U_hat']:.5f}")
print(f"stat = {res_ef_trade['stat']:.3f}   q = {res_ef_trade['q_crit']:.3f}   p-value = {res_ef_trade['p_value']:.4f}")
print(f"Reject H₀ : {res_ef_trade['reject']}")


## 6. Eigenfunction test — break 2020-03-10 (COVID)

With Δv = 0.01 (minimal "relevant" rotation threshold).

In [ ]:
res_ef_covid = eigenfunction_test(
    Y,
    j          = 1,
    Delta_v    = 0,
    alpha      = 0.05,
    K          = 20,
    break_date = "2020-03-10",
    plot       = True,
    pair_name  = "panel_ef_covid",
    dates      = days,
    delta=delta
)

print(f"Break : 2020-03-10")
print(f"D̂  = {res_ef_covid['D_hat']:.5f}   Û = {res_ef_covid['U_hat']:.5f}")
print(f"stat = {res_ef_covid['stat']:.3f}   q = {res_ef_covid['q_crit']:.3f}   p-value = {res_ef_covid['p_value']:.4f}")
print(f"Reject H₀ : {res_ef_covid['reject']}")


## 7. Tests on the global break — 2022-02-17 (invasion of Ukraine)

The global-operator CUSUM identifies Feb. 2022 as the dominant break.
We test whether this break is carried by τ₁, v₁, or both.

In [ ]:
res_ev_ukraine = eigenvalue_test(
    Y, j=1, Delta_tau=0.1, alpha=0.05, K=20,
    break_date = cp_global["break_date"],
    plot=True, pair_name="panel_ev_ukraine", dates=days,delta=delta
)

res_ef_ukraine = eigenfunction_test(
    Y, j=1, Delta_v=0.1, alpha=0.05, K=20,
    break_date = cp_global["break_date"],
    plot=True, pair_name="panel_ef_ukraine", dates=days,delta=delta
)

print(f"Break : {str(cp_global['break_date'])[:10]}")
print(f"[Eigenvalue]    τ^(1)={res_ev_ukraine['tau1_j']:.5f}  τ^(2)={res_ev_ukraine['tau2_j']:.5f}  E^={res_ev_ukraine['E_hat']:.5f}  V^={res_ev_ukraine['V_hat']:.5f}. stat={res_ev_ukraine['stat']:.3f}  p={res_ev_ukraine['p_value']:.4f}  Reject={res_ev_ukraine['reject']}")
print(f"[Eigenfunction] D̂={res_ef_ukraine['D_hat']:.5f}  Û={res_ef_ukraine['U_hat']:.5f}  stat={res_ef_ukraine['stat']:.3f}  p={res_ef_ukraine['p_value']:.4f}  Reject={res_ef_ukraine['reject']}")


## 8. Calibrating the relevance threshold (calm period 2014–2016)

Δv = 95% quantile of ‖Δv̂₁‖²_L² between adjacent blocks in a period without known crisis.
Δτ = 95% quantile of (Δτ̂₁)² between adjacent blocks.

In [ ]:
cal_v = calibrate_delta_v(
    Y, j=1,
    calm_start="2014-01-01", calm_end="2016-12-31",
    dates=days, n_blocks=10, q=0.95, method="empirical",
)
cal_tau = calibrate_delta_tau(
    Y, j=1,
    calm_start="2014-01-01", calm_end="2016-12-31",
    dates=days, n_blocks=10, q=0.95,
)

print(f"Δv  calibrated (q=0.95) = {cal_v['delta_v']:.6f}")
print(f"  distances range : [{cal_v['distances'].min():.4f}, {cal_v['distances'].max():.4f}]")
print()
print(f"Δτ  calibrated (q=0.95) = {cal_tau['delta_tau']:.6f}")
print(f"  sq_diffs range  : [{cal_tau['sq_diffs'].min():.6f}, {cal_tau['sq_diffs'].max():.6f}]")


## 9. Tests with calibrated thresholds

With Δv and Δτ calibrated on the calm period, a break must exceed
the natural variation of the reference period to be "relevant".

In [ ]:
# Eigenvalue test with calibrated Δτ — Brexit break
res_ev_cal = eigenvalue_test(
    Y, j=1, Delta_tau=cal_tau["delta_tau"], alpha=0.05, K=20,
    break_date=cp_ev["break_date"], dates=days,delta=delta
)

# Eigenfunction test with calibrated Δv — trade-war break
res_ef_cal = eigenfunction_test(
    Y, j=1, Delta_v=cal_v["delta_v"], alpha=0.05, K=20,
    break_date=cp_ef["break_date"], dates=days,delta=delta
)

print("=== Eigenvalue test (calibrated threshold Δτ = {:.4f}) ===".format(cal_tau["delta_tau"]))
print(f"  Break {str(cp_ev['break_date'])[:10]} | Ê={res_ev_cal['E_hat']:.6f}  stat={res_ev_cal['stat']:.3f}  p={res_ev_cal['p_value']:.4f}  Reject={res_ev_cal['reject']}")
print()
print("=== Eigenfunction test (calibrated threshold Δv = {:.4f}) ===".format(cal_v["delta_v"]))
print(f"  Break {str(cp_ef['break_date'])[:10]} | D̂={res_ef_cal['D_hat']:.6f}  stat={res_ef_cal['stat']:.3f}  p={res_ef_cal['p_value']:.4f}  Reject={res_ef_cal['reject']}")
